# Lecture 32 - Capstone Project: End-to-End Data Science Pipeline

## Learning Objectives

- Load a dataset and perform initial cleaning
- Conduct univariate and multivariate EDA
- Engineer features through encoding, scaling, and creation
- Train and compare multiple models
- Tune hyperparameters and select the best model
- Communicate findings with narrative, plots, and model summary

## Key Topics

- Load and clean: handle missing values, outliers, type fixes
- EDA: univariate and multivariate analysis, visualisations
- Feature engineering: encoding, scaling, new feature creation
- Modelling: train multiple models, tune hyperparameters
- Evaluation: compare models, select best, interpret results
- Communication: final notebook with narrative, plots, summary

## Capstone: End-to-End Data Science Pipeline

This capstone brings together everything you have learned in Phase 4. We will work through a complete data science pipeline on the **Wine Quality** dataset from scikit-learn, progressing from raw data to a final model evaluation.

The pipeline will cover:
1. **Load & Clean**: inspect the data, handle missing values and outliers
2. **EDA**: explore distributions, correlations, and relationships
3. **Feature Engineering**: scale, encode, and create new features
4. **Modelling**: train RandomForest and LogisticRegression with cross-validation
5. **Hyperparameter Tuning**: find the best configuration for each model
6. **Evaluation & Comparison**: compare models and select the winner
7. **Communication**: present results with clear narrative and visuals

This is the workflow used in real-world data science projects every day.

In [ ]:
# Step 1: Load and explore the Wine Quality dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine

wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df["target"] = wine.target
target_names = wine.target_names

print(f"Dataset shape: {df.shape}")
print(f"Feature names: {list(wine.feature_names)}")
print(f"Target classes: {target_names}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nData types:")
print(df.dtypes)

In [ ]:
# Step 2: Initial cleaning — check missing values and outliers
print("Missing values:")
print(df.isnull().sum())

print("\nBasic statistics:")
print(df.describe())

# Check for outliers using IQR
def count_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return ((series < (Q1 - 1.5 * IQR)) | (series > (Q3 + 1.5 * IQR))).sum()

outlier_counts = df.drop("target", axis=1).apply(count_outliers)
print("\nOutlier counts per feature:")
print(outlier_counts[outlier_counts > 0])

## Exploratory Data Analysis

EDA is the most important step in any data science project. We will:

1. **Univariate analysis**: examine the distribution of each feature and the target
2. **Multivariate analysis**: explore correlations between features and relationships with the target

These visualisations help us understand which features might be predictive, whether there are data quality issues, and what transformations might be needed.

The Wine Quality dataset has 13 chemical features (alcohol, malic acid, ash, etc.) and a target with 3 classes of wine cultivar. This is a multiclass classification problem.

In [ ]:
# Step 3: Univariate analysis — target distribution and feature histograms
plt.figure(figsize=(14, 6))

# Target distribution
plt.subplot(1, 2, 1)
target_counts = df["target"].value_counts().sort_index()
plt.bar(target_names, target_counts.values)
plt.title("Target Class Distribution")
plt.ylabel("Count")

# Feature distributions
plt.subplot(1, 2, 2)
df.drop("target", axis=1).hist(bins=20, figsize=(12, 10))
plt.suptitle("Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Step 4: Multivariate analysis — correlations and pairplots
plt.figure(figsize=(12, 10))
corr = df.drop("target", axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=0.5)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

# Pairplot of selected features (to avoid overcrowding)
selected_features = ["alcohol", "malic_acid", "ash", "color_intensity", "proline"]
sns.pairplot(df, vars=selected_features, hue="target", palette="Set1")
plt.show()

## Feature Engineering

Feature engineering transforms raw features into better inputs for our models. For this dataset:

- **Scaling**: many ML algorithms (LogisticRegression with regularisation, KNN) require features on the same scale. We use StandardScaler.
- **New features**: we can create interaction terms (e.g., alcohol × color_intensity) and polynomial features to capture non-linear relationships.
- **Feature selection**: based on our correlation analysis and feature importance from tree-based models, we may drop or keep features.

We will use a `ColumnTransformer` and `Pipeline` to keep our preprocessing clean and prevent data leakage.

In [ ]:
# Step 5: Feature engineering — scaling and new features
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

# Create preprocessing pipeline
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully")
print(f"Mean after scaling: {X_train_scaled.mean(axis=0).round(3)[:5]}...")
print(f"Std after scaling:  {X_train_scaled.std(axis=0).round(3)[:5]}...")

In [ ]:
# Create interaction features
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

print(f"Original features: {X_train_scaled.shape[1]}")
print(f"After interactions: {X_train_poly.shape[1]}")
print(f"Interaction feature names:")
print(poly.get_feature_names_out()[:15], "...")

## Modelling: Train and Compare Multiple Models

A good data scientist never settles on one model. We train multiple families of models and compare them using cross-validation:

- **LogisticRegression**: strong linear baseline, well-calibrated probabilities
- **RandomForestClassifier**: handles non-linear relationships, robust to outliers
- **KNeighborsClassifier**: simple, non-parametric baseline

We then tune the most promising model using `GridSearchCV` to find optimal hyperparameters.

All training happens inside a `Pipeline` that includes scaling, so we never leak information from the test set during training.

In [ ]:
# Step 6: Train and compare multiple models with cross-validation
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings("ignore")

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

results = []
for name, model in models.items():
    # Create pipeline with scaling for each model
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])
    scores = cross_val_score(pipe, X, y, cv=5, scoring="accuracy")
    results.append({
        "Model": name,
        "Mean CV Accuracy": scores.mean(),
        "Std CV Accuracy": scores.std()
    })
    print(f"{name:25s}  CV accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")

results_df = pd.DataFrame(results).sort_values("Mean CV Accuracy", ascending=False)

In [ ]:
# Step 7: Hyperparameter tuning with GridSearchCV
from sklearn.model_selection import GridSearchCV

pipe_rf = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestClassifier(random_state=42))
])

param_grid = {
    "rf__n_estimators": [50, 100, 200],
    "rf__max_depth": [None, 5, 10, 15],
    "rf__min_samples_split": [2, 5, 10]
}

grid_search = GridSearchCV(
    pipe_rf, param_grid, cv=5, scoring="accuracy", n_jobs=-1, verbose=0
)
grid_search.fit(X, y)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.3f}")

# Store best model
best_rf = grid_search.best_estimator_

In [ ]:
# Also tune LogisticRegression
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, random_state=42))
])

lr_param_grid = {
    "lr__C": [0.01, 0.1, 1, 10, 100],
    "lr__solver": ["lbfgs", "liblinear"]
}

grid_lr = GridSearchCV(
    pipe_lr, lr_param_grid, cv=5, scoring="accuracy", n_jobs=-1
)
grid_lr.fit(X, y)

print(f"Best LR parameters: {grid_lr.best_params_}")
print(f"Best LR CV accuracy: {grid_lr.best_score_:.3f}")
best_lr = grid_lr.best_estimator_

## Model Evaluation and Selection

After tuning, we evaluate the best models on the held-out test set. We use:

- **Accuracy**: overall correctness
- **Confusion matrix**: where does the model get confused between classes?
- **Classification report**: precision, recall, F1 per class
- **Feature importance** (for RandomForest): which features drive predictions?

The final model is chosen based on test set performance, simplicity, and interpretability. Sometimes the slightly less accurate model is preferred because it is simpler to explain and deploy.

In [ ]:
# Step 8: Final evaluation on test set
from sklearn.metrics import confusion_matrix, classification_report

# Evaluate best Random Forest
y_pred_rf = best_rf.predict(X_test)

print("=== Random Forest (Tuned) ===")
print(f"Test accuracy: {best_rf.score(X_test, y_test):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=target_names))

# Confusion matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names)
plt.title("Confusion Matrix - Random Forest (Test Set)")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.show()

In [ ]:
# Compare with tuned LogisticRegression
y_pred_lr = best_lr.predict(X_test)

print("=== Logistic Regression (Tuned) ===")
print(f"Test accuracy: {best_lr.score(X_test, y_test):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=target_names))

cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt="d", cmap="Greens",
            xticklabels=target_names, yticklabels=target_names)
plt.title("Confusion Matrix - Logistic Regression (Test Set)")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.show()

In [ ]:
# Feature importance from the best Random Forest
rf_model = best_rf.named_steps["rf"]
feat_imp = pd.DataFrame({
    "feature": wine.feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

plt.figure(figsize=(10, 5))
plt.barh(feat_imp["feature"], feat_imp["importance"])
plt.xlabel("Feature Importance")
plt.title("Top Predictive Features - Wine Quality Dataset")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 5 features:")
print(feat_imp.head().to_string(index=False))

## Final Summary and Communication

This capstone demonstrated a complete end-to-end data science pipeline:

1. **Data Loading & Cleaning**: inspected the Wine Quality dataset, checked for missing values and outliers
2. **EDA**: visualised feature distributions, class balance, and correlations; identified key relationships
3. **Feature Engineering**: standardised features and created interaction terms
4. **Modelling**: compared LogisticRegression, RandomForest, and KNN with cross-validation
5. **Hyperparameter Tuning**: used GridSearchCV to optimise RandomForest and LogisticRegression
6. **Evaluation**: compared tuned models on a held-out test set with accuracy, confusion matrices, and classification reports

Both tuned Random Forest and Logistic Regression performed well on this dataset. The final choice depends on whether you prioritise interpretability (LogisticRegression provides clear coefficients) or predictive power (RandomForest captures non-linear interactions automatically). Always complement model performance with domain knowledge when making the final selection.

In [ ]:
# Final comparison table
comparison = pd.DataFrame({
    "Model": ["RandomForest (tuned)", "LogisticRegression (tuned)"],
    "Test Accuracy": [
        best_rf.score(X_test, y_test),
        best_lr.score(X_test, y_test)
    ],
    "Best CV Accuracy": [
        grid_search.best_score_,
        grid_lr.best_score_
    ]
})
print("=== Final Model Comparison ===")
print(comparison.to_string(index=False))

# Determine winner
winner = comparison.loc[comparison["Test Accuracy"].idxmax(), "Model"]
print(f"\nWinner: {winner}")

In [ ]:
# Summary visualisation
models_compared = ["LR (tuned)", "RF (tuned)"]
accuracies = [
    best_lr.score(X_test, y_test),
    best_rf.score(X_test, y_test)
]
cv_scores = [
    grid_lr.best_score_,
    grid_search.best_score_
]

x = np.arange(len(models_compared))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, accuracies, width, label="Test Accuracy", color="steelblue")
plt.bar(x + width/2, cv_scores, width, label="Best CV Accuracy", color="coral")
plt.ylabel("Accuracy")
plt.title("Model Performance Comparison")
plt.xticks(x, models_compared)
plt.ylim(0.8, 1.0)
plt.legend()
plt.grid(axis="y", alpha=0.3)

# Add value labels
for i, (acc, cv) in enumerate(zip(accuracies, cv_scores)):
    plt.text(i - width/2, acc + 0.005, f"{acc:.3f}", ha="center", fontsize=10)
    plt.text(i + width/2, cv + 0.005, f"{cv:.3f}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

## Data Science Connection

You have now completed the full data science lifecycle — from raw data to a tuned, evaluated model. This capstone pipeline is the same process used by data scientists at companies of all sizes: understand the data, clean it, explore it, engineer features, model, evaluate, and communicate. The Wine Quality dataset is a classic benchmark; the skills you applied here transfer directly to real-world problems. Congratulations on completing Phase 4 — you are ready to apply these techniques to your own datasets and projects.